# 4. Spells: targeting, resolution, and honest refusal

Until recently every instant and sorcery in this engine resolved and did
*nothing*, and said so. This notebook covers the layer that changed that — and,
just as importantly, how the engine refuses what it still cannot do.

The refusal is the interesting part. An engine that guesses at a card it does not
understand produces games that look completely reasonable and are wrong.

In [1]:
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "data" / "cards.duckdb").exists() and REPO != REPO.parent:
    REPO = REPO.parent

from manalab.cards.mana import parse_cost
from manalab.cards.model import Card
from manalab.cards.store import CardStore

store = CardStore(REPO / "data" / "cards.duckdb")


def creature(name, power, toughness, *keywords, cost="{1}{G}"):
    # A synthetic creature, so a demo shows a rule and not a card.
    return Card(name.lower().replace(" ", "-"), name, parse_cost(cost),
                "Creature - Test", power=str(power), toughness=str(toughness),
                keywords=frozenset(keywords))


FOREST = store.by_name("Forest")

In [2]:
from manalab.engine.zones import Zone


def show(state, title=""):
    # Print the whole visible game state. The workhorse of these notebooks.
    if title:
        print(f"--- {title}")
    print(f"    turn {state.turn}, active P{state.active_player}")
    for player in state.players:
        flags = " LOST" if player.has_lost else ""
        print(f"    P{player.index}: {player.life} life{flags}"
              f"   hand={len(state.hand(player.index))}"
              f"  library={len(state.library(player.index))}"
              f"  graveyard={len(state.graveyard(player.index))}")
        for obj in state.battlefield(player.index):
            marks = []
            if obj.tapped:
                marks.append("tapped")
            if obj.damage:
                marks.append(f"dmg{obj.damage}")
            if obj.deathtouched:
                marks.append("deathtouched")
            if obj.summoning_sick:
                marks.append("sick")
            if obj.attacking is not None:
                marks.append(f"attacking P{obj.attacking}")
            if obj.blocking is not None:
                marks.append("blocking")
            if obj.counters:
                marks.extend(f"{k}x{v}" for k, v in sorted(obj.counters.items()))
            body = f"{obj.power}/{obj.toughness}" if obj.is_creature else "land"
            suffix = f"  [{' '.join(marks)}]" if marks else ""
            print(f"        #{obj.id} {obj.card.name:<22} {body:<6}{suffix}")
    stack = state.zone(Zone.STACK)
    if stack:
        print(f"    stack (top last): {[o.card.name for o in stack]}")

## Cards are data plus effect verbs

A card is never modelled as "a creature with power and toughness". It is Scryfall
data plus a tuple of `Effect(verb, params)`, parsed from the oracle text. Growing
the engine means implementing more verbs — not rewriting the model.

In [3]:
for name in ["Lightning Bolt", "Murder", "Divination", "Giant Growth", "Counterspell"]:
    card = store.by_name(name)
    print(f"{name:<16} {card.oracle_text}")
    for effect in card.effects:
        print(f"      {effect.verb:<14} {dict(effect.params)}")
    print()

Lightning Bolt   Lightning Bolt deals 3 damage to any target.
      deal_damage    {'amount': 3, 'target': 'any target'}

Murder           Destroy target creature.
      destroy        {'target': 'target creature'}

Divination       Draw two cards.
      draw           {'amount': 2}

Giant Growth     Target creature gets +3/+3 until end of turn.
      pump           {'target': 'Target creature', 'amount': '+3/+3'}

Counterspell     Counter target spell.
      counter_spell  {'target': 'spell'}



## The target language, and what it refuses

Targets arrive as English. `engine.targets.parse` recognises a small set exactly
and returns `None` for everything else — which every caller treats as "this card
is not modelled".

That refusal is deliberate. Guessing that `"target nonblack creature"` means "any
creature" would make illegal plays legal, and the resulting win rates would look
perfectly sensible.

In [4]:
from manalab.engine import targets as T

phrases = ["any target", "target creature", "target creature you control",
           "target player", "target nonblack creature",
           "target creature an opponent controls", "all creatures",
           "target attacking or blocking creature"]
for phrase in phrases:
    spec = T.parse(phrase)
    verdict = spec.kind.value if spec else "REFUSED"
    print(f"  {phrase:<40} {verdict}")

  any target                               any
  target creature                          creature
  target creature you control              creature_you_control
  target player                            player
  target nonblack creature                 REFUSED
  target creature an opponent controls     REFUSED
  all creatures                            REFUSED
  target attacking or blocking creature    REFUSED


## Enumerating legal targets

Same contract as blocking: the engine lists the legal choices and the agent picks
one. It cannot invent a target.

In [5]:
from manalab.engine.state import GameState

state = GameState.new([[], []], seed=0)
mine = state.create(creature("Mine", 2, 2), owner=0, zone=Zone.BATTLEFIELD)
theirs = state.create(creature("Theirs", 4, 4), owner=1, zone=Zone.BATTLEFIELD,
                      controller=1)

for kind in (T.TargetKind.CREATURE, T.TargetKind.CREATURE_YOU_CONTROL,
             T.TargetKind.ANY, T.TargetKind.OPPONENT):
    found = T.legal_targets(state, 0, T.TargetSpec(kind))
    print(f"  {kind.value:<22} {[t.describe(state) for t in found]}")

  creature               ['Mine', 'Theirs']
  creature_you_control   ['Mine']
  any                    ['Mine', 'Theirs', 'P0', 'P1']
  opponent               ['P1']


## A spell with no legal target cannot be cast

CR 601.2c. Without this, a removal spell gets "cast" into an empty board and
fizzles, which reads as a play mistake rather than an engine bug.

In [6]:
from manalab.engine import actions as A
from manalab.engine.steps import Step

SWAMP = store.by_name("Swamp")
murder = store.by_name("Murder")

def castable(opponent_creatures):
    state = GameState.new([[], []], seed=0)
    state.active_player = 0
    for _ in range(3):
        state.create(SWAMP, owner=0, zone=Zone.BATTLEFIELD)
    state.create(murder, owner=0, zone=Zone.HAND)
    for i in range(opponent_creatures):
        state.create(creature(f"Target{i}", 2, 2), owner=1,
                     zone=Zone.BATTLEFIELD, controller=1)
    options = A.legal(state, 0, Step.PRECOMBAT_MAIN, stack_empty=True)
    return [o for o in options if isinstance(o, A.CastSpell)], state

for n in (0, 1, 3):
    casts, state = castable(n)
    described = [c.describe(state) for c in casts]
    print(f"  {n} creature(s) on board -> {len(casts)} castings  {described}")

  0 creature(s) on board -> 0 castings  []
  1 creature(s) on board -> 1 castings  ['cast Murder -> Target0']
  3 creature(s) on board -> 3 castings  ['cast Murder -> Target0', 'cast Murder -> Target1', 'cast Murder -> Target2']


## Resolution

Targets are chosen on announcement and **re-checked on resolution** (CR 608.2b).
A spell whose target left in response does nothing.

In [7]:
from manalab.engine import resolve, sba

state = GameState.new([[], []], seed=0)
victim = state.create(creature("Victim", 2, 2), owner=1, zone=Zone.BATTLEFIELD,
                      controller=1)
bolt = state.create(store.by_name("Lightning Bolt"), owner=0, zone=Zone.STACK)

log = []
resolve.apply(state, bolt, (T.Target(object_id=victim.id),), log)
sba.check(state)
print("  log:", log)
print("  opponent battlefield:", [o.card.name for o in state.battlefield(1)])

  log: ['Lightning Bolt deals 3 to Victim']
  opponent battlefield: []


In [8]:
# Now the same bolt, but the target is gone before it resolves.
state = GameState.new([[], []], seed=0)
victim = state.create(creature("Victim", 2, 2), owner=1, zone=Zone.BATTLEFIELD,
                      controller=1)
bolt = state.create(store.by_name("Lightning Bolt"), owner=0, zone=Zone.STACK)
target = T.Target(object_id=victim.id)

state.move(victim, Zone.GRAVEYARD)   # it died in response
log = []
resolve.apply(state, bolt, (target,), log)
print("  log:", log)
print("  the opponent took no damage:", state.players[1].life)

  log: ['Lightning Bolt fizzles: target is gone']
  the opponent took no damage: 20


In [9]:
# The other verbs, each on a real card.
def demo(card_name, setup):
    state = GameState.new([[], []], seed=0)
    targets_chosen = setup(state)
    spell = state.create(store.by_name(card_name), owner=0, zone=Zone.STACK)
    log = []
    resolve.apply(state, spell, targets_chosen, log)
    sba.check(state)
    return state, log

def with_library(state):
    for _ in range(5):
        state.create(creature("Card", 1, 1), owner=0, zone=Zone.LIBRARY)
    return ()

state, log = demo("Divination", with_library)
print("  Divination ->", log, "| hand size", len(state.hand(0)))

def with_own_creature(state):
    obj = state.create(creature("Mine", 2, 2), owner=0, zone=Zone.BATTLEFIELD)
    return (T.Target(object_id=obj.id),)

state, log = demo("Giant Growth", with_own_creature)
obj = state.battlefield(0)[0]
print(f"  Giant Growth -> {log} | now {obj.power}/{obj.toughness}")

  Divination -> ['Divination: P0 draws 2'] | hand size 2
  Giant Growth -> ['Giant Growth pumps Mine +3/+3'] | now 5/5


## Fidelity: the engine's account of its own blind spots

`GameResult.unmodeled` reports blind spots *after* a game. `engine.fidelity`
answers the same question *before* one — which is what lets a deck search
restrict itself to a pool it can trust.

It lives in the engine because the engine is the only thing that knows what it
implements. A second list anywhere else drifts.

In [10]:
from manalab.engine import fidelity

for name in ["Lightning Bolt", "Murder", "Divination", "Giant Growth",
             "Counterspell", "Doom Blade", "Craterhoof Behemoth",
             "Birds of Paradise", "Llanowar Elves", "Leatherback Baloth"]:
    report = fidelity.assess(store.by_name(name))
    mark = "OK " if report.faithful else "NO "
    print(f"  {mark} {name:<22} {'; '.join(report.reasons) or 'fully modelled'}")

  OK  Lightning Bolt         fully modelled
  OK  Murder                 fully modelled
  OK  Divination             fully modelled
  OK  Giant Growth           fully modelled
  NO  Counterspell           verb not implemented: counter_spell
  NO  Doom Blade             target not understood: 'target nonblack creature'
  NO  Craterhoof Behemoth    rules text the engine does not execute
  NO  Birds of Paradise      rules text the engine does not execute
  OK  Llanowar Elves         fully modelled
  OK  Leatherback Baloth     fully modelled


Note **Birds of Paradise** is refused while **Llanowar Elves** is accepted. Birds
adds mana of *any* colour and the engine only understands a source that taps for
one fixed colour. Both are one-mana mana creatures; only one of them is modelled
correctly, and a search told otherwise would rank them identically.

## What this buys, measured

In [11]:
from manalab.evaluate import pool as P

print(f"{'colour':<9}{'faithful':>10}{'set aside':>11}   commonest reason for refusal")
for code, name in {"W": "white", "U": "blue", "B": "black",
                   "R": "red", "G": "green"}.items():
    pool = P.mono_color(store, code, fmt="modern")
    top = pool.gaps()[0][0] if pool.gaps() else "-"
    print(f"  {name:<7}{pool.size:>10}{len(pool.text_tier):>11}   {top}")

colour     faithful  set aside   commonest reason for refusal


  white         187       3136   rules text the engine does not execute


  blue          122       3123   rules text the engine does not execute


  black         137       3166   rules text the engine does not execute


  red           151       3121   rules text the engine does not execute


  green         187       3009   rules text the engine does not execute


## The honest state of the engine

**Implemented:** zones and the move primitive, the twelve steps, priority, the
stack, sorcery-speed timing, mana payment with colour matching, combat with
evasion / multi-blocks / trample / deathtouch / first and double strike /
lifelink / vigilance, state-based actions, and five resolution verbs
(`deal_damage`, `destroy`, `draw`, `gain_life`, `pump`) with targeting.

**Not implemented, and refused rather than approximated:** triggered abilities,
activated abilities, continuous effects and anthems, counterspells, tokens,
counters as a card mechanic, the legend rule, protection and hexproof, flash,
and every target phrase outside the four the parser recognises.

That second list is the roadmap. `pool.gaps()` ranks it by how many cards each
gap costs — which is the only sensible order to work in.